# XGBoost Fraud Detection Model

This notebook trains an XGBoost classifier using both original dataset variables and engineered fraud indicators.

The objective is to improve fraud recall while maintaining reasonable precision on a highly imbalanced fraud detection dataset.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/DataSet.csv")

In [ ]:
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix
)

## Feature Engineering

Several fraud indicators and interaction features were created based on patterns discovered during exploratory analysis.

These engineered features expose high-risk regions of the feature space that may not be fully captured by the original variables alone.

In [ ]:
df["F115_high"] = (df["F115"] > 0.78).astype(int)

df["F2582_hot"] = (
    (df["F2582"] > -0.03)
    & (df["F2582"] <= 0)
).astype(int)

df["F2956_hot"] = (
    (df["F2956"] > 19)
    & (df["F2956"] <= 32)
).astype(int)

df["F531_hot"] = (
    (df["F531"] > 0.95)
    & (df["F531"] <= 1.35)
).astype(int)

df["F2737_safe"] = (
    (df["F2737"] > 0)
    & (df["F2737"] <= 0.04)
).astype(int)

df["F2582_F531"] = (
    df["F2582_hot"]
    & df["F531_hot"]
).astype(int)

df["fraud_cluster_1"] = (
    df["F2582_hot"]
    & df["F2956_hot"]
    & df["F531_hot"]
).astype(int)

df["f3836_hot"] = (
    (df["F3836"] > 148.596 ) &
    (df["F3836"] <= 20077.212)
).astype(int)

df["F2956_F115"] = (
    df["F2956_hot"] &
    df["F115_high"]
).astype(int)

df["F2582_pos_F2956_low"] = (
    (df["F2582"] > 0.15) &
    (df["F2956"] < 60)
).astype(int)

## Feature Selection

The final feature set consists of original dataset variables, engineered fraud indicators, interaction features, and missing-value indicators derived during exploratory analysis.

These features were selected based on fraud-rate analysis, feature importance investigation, and model performance.

In [ ]:
features = [
    "F115",
    "F2582",
    "F2956",
    "F531",
    "F2737",
    "F670",
    "F673",
    "F3891",
    "F3889",

    # engineered features

    "F115_high",
    "F2582_hot",
    "F2956_hot",
    "F531_hot",
    "F2737_safe",
    "F2582_F531",
    "fraud_cluster_1",
    "f3836_hot",
    "F2956_F115",
    "F2582_pos_F2956_low"

]

family_cols = [
    "F664","F665","F666",
    "F667","F668","F669",
    "F670","F671","F672",
    "F673","F674","F675",

    "F1","F2","F3","F4","F5","F6","F7","F8","F9","F10","F12"
]

for col in family_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

features += [f"{col}_missing" for col in family_cols]

In [ ]:
X = pd.get_dummies(
    df[features],
    columns=["F3891", "F3889"],
    dummy_na=True
)

y = df["F3924"]

## Data Preparation

Categorical variables are one-hot encoded and missing values are handled prior to model training.

A stratified train-test split is used to preserve fraud prevalence across datasets.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train = X_train.fillna(-999)
X_test = X_test.fillna(-999)

## Model Training

XGBoost is trained using class weighting to address severe class imbalance.

The model is optimized for fraud detection performance rather than overall accuracy.

In [ ]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=110,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

## Model Evaluation

Performance is evaluated across multiple probability thresholds.

Because fraudulent accounts represent a small minority of observations, recall is prioritized over accuracy.

In [ ]:
y_prob = xgb.predict_proba(X_test)[:, 1]

for threshold in [0.5, 0.3, 0.2, 0.1]:

    y_pred = (y_prob >= threshold).astype(int)

    print(f"\nThreshold: {threshold}")

    print(confusion_matrix(y_test, y_pred))

    print(classification_report(y_test, y_pred))

In [ ]:
for threshold in [0.25, 0.30, 0.35, 0.40, 0.45]:
    y_pred = (y_prob >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    print(
        f"Threshold={threshold:.2f} | "
        f"TP={tp} FP={fp} FN={fn} | "
        f"Precision={precision:.3f} Recall={recall:.3f}"
    )

## Feature Importance Analysis

Feature importance scores provide insight into which variables contribute most strongly to fraud predictions.

In [ ]:
importance = pd.Series(
    xgb.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importance.head(15)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

importance.head(15).sort_values().plot.barh()

plt.title("Top 15 XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

plt.savefig("../figures/xgb_feature_importance.png",
            bbox_inches="tight")

## Ranking Performance

Average Precision evaluates the model's ability to rank fraudulent accounts ahead of legitimate accounts independent of a fixed threshold.

In [ ]:
from sklearn.metrics import average_precision_score

ap = average_precision_score(y_test, y_prob)

print("Average Precision:", ap)

## Exporting Predictions

Predicted probabilities are saved for later comparison with other models and ensemble construction.

In [ ]:
xgb_probs = pd.DataFrame({
    "prob": y_prob
}, index=X_test.index)

xgb_probs.to_csv("../data/processed/xgb_probs.csv")

In [ ]:
pd.DataFrame({
    "actual": y_test
}).to_csv("../data/processed/y_test.csv")

## Key Findings

The XGBoost model significantly improved fraud detection performance compared to the CatBoost baseline.

Key observations:

- Engineered fraud indicators contributed substantially to model performance.
- Missing-value indicators proved highly informative.
- The model achieved strong fraud recall across multiple thresholds.
- Feature importance analysis consistently highlighted F2737, F531, F2956, F2582, and F115 as important predictors.

The predicted probabilities generated in this notebook are later used for ensemble construction and comparison with LightGBM and Neural Network models.